Imports

In [ ]:
!pip install lightly #good library for contrastive learning based on google search
!pip install scikit-learn

In [ ]:
# Fix for sympy-torch incompatibility
!pip uninstall -y sympy
!pip install sympy==1.14.0

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from cub2011 import Cub2011

Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)


Transforms, NT_xent_loss, Dataset class and model class

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

simclr_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

evaluation_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

def nt_xent_loss(z_i, z_j, temperature=0.05):
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)

    similarity = torch.matmul(z, z.T)
    N = z_i.shape[0]

    mask = (~torch.eye(2*N, dtype=bool)).to(z.device)
    sim = similarity / temperature
    exp_sim = torch.exp(sim) * mask

    positive_sim = torch.exp(F.cosine_similarity(z_i, z_j) / temperature)
    positives = torch.cat([positive_sim, positive_sim], dim=0)

    denominator = exp_sim.sum(dim=1)
    loss = -torch.log(positives / denominator)
    return loss.mean()

class SimCLRDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.dataset = base_dataset
        self.transform = transform

    def __getitem__(self, index):
        image, _ = self.dataset[index]
        xi = self.transform(image)
        xj = self.transform(image)
        return xi, xj

    def __len__(self):
        return len(self.dataset)

class SimCLRModel(nn.Module):
    def __init__(self, model_type, projection_dim=128):
        super().__init__()
        base_model = model_type(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        num_ftrs = base_model.fc.in_features
        base_model.fc = nn.Identity()
        self.encoder = base_model
        self.projection_head = nn.Sequential(
            nn.Linear(num_ftrs, 2048),
            nn.ReLU(),
            nn.Linear(2048, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

def extract_embeddings_labels(model, loader):
    embeddings, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            z = model(x)
            embeddings.append(z.cpu())
            labels.append(y)
    return torch.cat(embeddings), torch.cat(labels)

training

In [ ]:
train_dataset = Cub2011(root=str('./cub2011'), train=True, download=True)
contrastive_dataset = SimCLRDataset(train_dataset, simclr_transform)
train_loader = DataLoader(contrastive_dataset, batch_size=128, shuffle=True, num_workers=2)

model = SimCLRModel(models.resnet50).to(device)

#https://discuss.pytorch.org/t/passing-to-the-optimizers-frozen-parameters/83358
#https://discuss.pytorch.org/t/how-can-i-exclude-some-parameters-in-optimizer-during-training/90208
#https://discuss.pytorch.org/t/best-practice-for-freezing-layers/58156
# Freeze the encoder weights, only train the projection head
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(model.parameters(), lr=.005)
#https://www.kozodoi.me/blog/gradient-accumulation-in-pytorch
#https://wandb.ai/wandb_fc/tips/reports/How-To-Implement-Gradient-Accumulation-in-PyTorch--VmlldzoyMjMwOTk5
accumulate_batches = 4


for epoch in range(50):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for i, samples in enumerate(tqdm(train_loader)):
        x_i, x_j = samples
        x_i = x_i.to(device)
        x_j = x_j.to(device)
        z_i = model(x_i)
        z_j = model(x_j)

        loss = nt_xent_loss(z_i, z_j)
        loss = loss/accumulate_batches
        loss.backward()

        if (i+1) % accumulate_batches == 0:
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulate_batches


    if(len(train_loader) % accumulate_batches != 0):
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    torch.save(model.state_dict(), "SimCLR_Resnet50_10e_512b_.001lr.pth")

print("Training completed!")

Files already downloaded and verified


100%|██████████| 47/47 [01:58<00:00,  2.53s/it]


Epoch 1 | Loss: 5.2478


100%|██████████| 47/47 [01:59<00:00,  2.55s/it]


Epoch 2 | Loss: 4.8591


...

Epoch 48 | Loss: 0.6008


100%|██████████| 47/47 [01:59<00:00,  2.54s/it]


Epoch 49 | Loss: 0.6135


100%|██████████| 47/47 [01:59<00:00,  2.55s/it]


Epoch 50 | Loss: 0.5783
Training completed!


In [ ]:
def recall_at_k(embeddings, labels, k=3):
    correct = 0
    numEmbeddings = len(labels)
    for i in range(numEmbeddings):
        current_embedding = embeddings[i]
        dist = torch.norm(embeddings - current_embedding, dim=1, p = None)
        dist[i] = float('inf')
        k_nearest = dist.topk(k, largest=False)
        for idx in k_nearest.indices:
            if labels[idx] == labels[i]:
                correct += 1
                break
    return correct / numEmbeddings

model = SimCLRModel(models.resnet50, projection_dim=128).to(device)
model.load_state_dict(torch.load("SimCLR_Resnet50_50e_512b_.001lr_extrafrozen.pth", map_location=device))
model.eval()

test_dataset = Cub2011(root='./cub2011', train=False, download=False, transform=evaluation_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

embeddings, labels = extract_embeddings_labels(model, test_loader)
r_at_1 = recall_at_k(embeddings, labels, k=1)
r_at_3 = recall_at_k(embeddings, labels, k=3)
r_at_5 = recall_at_k(embeddings, labels, k=5)
r_at_10 = recall_at_k(embeddings, labels, k=10)
print(f"R@1: {r_at_1:.4f}, R@3: {r_at_3:.4f}, R@5: {r_at_5:.4f}, R@10: {r_at_10:.4f}")


R@1: 0.2142, R@3: 0.3775, R@5: 0.4753, R@10: 0.5942
